# Intro
Finding a good flat which is near to your work place and is also near to e.g. the kindergarden/school of your kids, your favorite park etc. can be very difficult. Unfortunately the existing search engines in Germany for apartments like Immoscout, Immowelt, Immonet don't support to compute the travel time for an apartment to some destinations. Here I want to show you how to use Immospider to do that.
## Immospider
Immospider is a python program that crawls the Immoscout24 website. It is based on ideas from http://mfcabrera.com/data_science/2015/01/17/ichbineinberliner.html and https://github.com/balzer82/immoscraper . But it is faster and more flexible. 

### Installation

Immospider is using the popular python framework https://scrapy.org/ . To install you need Python 3. Then you can clone this repository and install the requirements via

    pip3 install -r requirements.txt
   
This should install scrapy and the googlemaps package for you. To use it you also need an API key for the googlemaps API. You should follow the instructions at https://github.com/googlemaps/google-maps-services-python#api-keys to get your API key.

### Usage

Let's assume you want to move to Berlin. You will work at some fancy startup near Alexanderplatz but your partner likes to go shopping at the KaDeWe. And you are searching for a flat with 2-3 rooms bigger than 60m^2 flat which should not be more expensive than 1000 Euro. You must enter these requirements in Immoscout24 website and search. If you search for whole Berlin you probably will find more than 500 results. As next step copy the url of your Immoscout search, because Immospider will use it. For the example given here the url is https://www.immobilienscout24.de/Suche/S-T/Wohnung-Miete/Berlin/Berlin/-/2,50-/60,00-/EURO--1000,00 . 
With this information you can now start Immospider like

    scrapy crawl immoscout -o apartments.csv -s GM_KEY=<Google Maps API Key> -a url=https://www.immobilienscout24.de/Suche/S-T/Wohnung-Miete/Berlin/Berlin/-/2,50-/60,00-/EURO--1000,00 -a dest="Alexanderplatz, Berlin" -a mode=transit -a dest2="KaDeWe, Berlin" -L INFO
    
The option `-o apartments.csv` specifies the output file. The parameter `-s GM_KEY=<Google Maps API Key>` sets your Google maps API key. The argument `dest="Alexanderplatz, Berlin" -a mode=transit` tells Immospider that you want to calculate the travel time for each apartment to Alexanderplatz using public transportation mode. The argument `dest2="KaDeWe, Berlin"` will additionaly compute the travel time via car (the default mode) to KaDeWe. You can have up to three destinations `dest1,dest2,dest3` and specify the mode for each destination `mode1,mode2,mode3`. The argument `-a url=...` must hold the search url from Immoscout. The optional parameter `-L INFO` can be added to generate more log output.

If you start Immospider with the given parameters here it might run up to 20 minutes, not because the crawler is slow, but because the Google Maps API takes some time to compute the travel time for each of the more than 500 apartments. If that is too slow for you, you should modify your search on Immoscout (and again copy the new url), so that the amount of search results is lower. If your result set is about 50 apartments, Immospider will only need 1-2 minutes to compute all the travel times. 

# Data Science
After Immospider has finished it is time for some data science. In the following we will use https://jupyter.org/ to analyze the result.

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('apartments.csv')

## Data Cleansing
We remove all the results without location (latitude, longitude).

In [ ]:
df.dropna(subset=["lng", "lat"], inplace=True)
df.head(n=10)

## Extracting a top10 list of apartments
We are searching for the apartments with the lowest travel time on average to our two destinations (Alexanderplatz and KaDeWe). To do this we compute the average travel time (avg_time) for each apartment and sort the list according to this value. Then we generate a list with the top10 results.

In [ ]:
df["avg_time"] = 0.5*(df.time_dest + df.time_dest2)
df.sort_values("avg_time", inplace=True)
top10=df.head(n=10)
top10

## Showing the results on a map
For better overview we show the results on a map. For this we use the package folium.

In [ ]:
# see https://nbviewer.jupyter.org/github/python-visualization/folium/blob/master/examples/Quickstart.ipynb
import folium
print(folium.__version__)

We create a map of Berlin and add a marker cluster to the map. Then we add our top 10 results as markers to the marker cluster. We also add a HTML popup to each result, showing the average travel time and a link to the expose.

In [ ]:
map = folium.Map(location=[52.520645, 13.409779])
# see http://deparkes.co.uk/2016/06/24/folium-marker-clusters/
marker_cluster = folium.MarkerCluster("appartments").add_to(map)

for index,row in enumerate(top10.itertuples()):
    html = '''{0}. <a target="_blank" href="{1}">{2}</a> </br>
    {3} </br>
    Average travel time: {4:.2f} min '''.format(index, row.url, row.title, row.address, row.avg_time)
    iframe = folium.element.IFrame(html=html.decode("utf-8").encode('ascii', 'xmlcharrefreplace'), width=300, height=100)
    popup = folium.Popup(iframe, max_width=300)
    folium.Marker([row.lat, row.lng], popup=popup).add_to(marker_cluster)

In [ ]:
# see https://nbviewer.jupyter.org/github/ocefpaf/folium_notebooks/blob/master/test_fit_bounds.ipynb
map.fit_bounds(map.get_bounds())
map